In [18]:
import json
from pathlib import Path
import chromadb
from tqdm import tqdm
import os
from groq import Groq

In [19]:
CHUNK_FILES = [
    "chunks_stategov.jsonl",
    "chunks_ustraveldocs.jsonl",
]

chunks = []
for path in CHUNK_FILES:
    p = Path(path)
    if not p.exists():
        print(f"⚠️  {path} not found — skipping")
        continue
    with p.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                chunks.append(json.loads(line))
    print(f"✅ Loaded {path}")

print(f"\n📊 Total chunks: {len(chunks)}")
print(f"Sample chunk:\n{json.dumps(chunks[0], indent=2)}")

✅ Loaded chunks_stategov.jsonl
✅ Loaded chunks_ustraveldocs.jsonl

📊 Total chunks: 279
Sample chunk:
{
  "heading": "Nonimmigrant Visa Categories",
  "text": "Nonimmigrant Visa Categories\nThe chart below contains many different purposes of temporary travel and the related nonimmigrant visa categories available on this website. Select a visa category below to learn more:",
  "content_type": "prose",
  "source_url": "https://travel.state.gov/content/travel/en/us-visas/visa-information-resources/all-visa-categories.html",
  "domain": "travel.state.gov",
  "category": "visa_types"
}


In [20]:
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

EMBED_MODEL = "all-MiniLM-L6-v2"
embed_fn = SentenceTransformerEmbeddingFunction(model_name=EMBED_MODEL)
print(f"✅ Model loaded: {EMBED_MODEL}")

✅ Model loaded: all-MiniLM-L6-v2


In [21]:
CHROMA_DIR      = "./chroma_db"
COLLECTION_NAME = "visa_rag"

client = chromadb.PersistentClient(path=CHROMA_DIR)
collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=embed_fn,
    metadata={"hnsw:space": "cosine"},
)
print(f"✅ Collection ready: '{COLLECTION_NAME}'")
print(f"   Vectors already in DB: {collection.count()}")

✅ Collection ready: 'visa_rag'
   Vectors already in DB: 279


In [22]:
BATCH_SIZE = 256

def make_doc_id(chunk, index):
    url  = chunk.get("source_url", "")[-60:]
    head = chunk.get("heading", "")[:30]
    part = chunk.get("split_part", 0)
    base = f"{url}|{head}|{part}"
    safe = "".join(c if c.isalnum() or c in "-_|" else "_" for c in base)
    return f"{safe}__{index}"

ids       = [make_doc_id(c, i) for i, c in enumerate(chunks)]
documents = [c["text"] for c in chunks]
metadatas = [
    {
        "heading":      c.get("heading", "")[:256],
        "content_type": c.get("content_type", ""),
        "source_url":   c.get("source_url", ""),
        "category":     c.get("category", ""),
        "domain":       c.get("domain", ""),
    }
    for c in chunks
]

for start in tqdm(range(0, len(chunks), BATCH_SIZE)):
    end = start + BATCH_SIZE
    collection.upsert(
        ids=ids[start:end],
        documents=documents[start:end],
        metadatas=metadatas[start:end],
    )

print(f"\n✅ Done! {collection.count()} vectors stored in '{CHROMA_DIR}/'")

100%|██████████| 2/2 [00:01<00:00,  1.63it/s]


✅ Done! 279 vectors stored in './chroma_db/'


In [23]:
# Check how many chunks were actually in your JSONL files
print(f"Chunks loaded in memory: {len(chunks)}")
print(f"Vectors in ChromaDB: {collection.count()}")

# Check split by source file
from collections import Counter
cats = Counter(c.get("category", "unknown") for c in chunks)
print(f"\nChunks per category:")
for cat, count in sorted(cats.items()):
    print(f"  {cat:<40} {count}")

# Check if both files were actually found
for path in CHUNK_FILES:
    p = Path(path)
    print(f"\n{path}: exists={p.exists()}", end="")
    if p.exists():
        lines = sum(1 for line in p.open() if line.strip())
        print(f"  lines={lines}")

Chunks loaded in memory: 279
Vectors in ChromaDB: 279

Chunks per category:
  administrative_processing                1
  ds160_application                        35
  photo_requirements                       14
  visa_denials                             29
  visa_fees                                40
  visa_process_ustraveldocs                97
  visa_types                               55
  visa_wait_times                          8

chunks_stategov.jsonl: exists=True  lines=182

chunks_ustraveldocs.jsonl: exists=True  lines=97


In [24]:
# Test retrieval quality across different query types
test_queries = [
    "what documents do i need for b2 tourist visa",
    "how much is the visa application fee",
    "what happens if my visa is denied",
    "how to fill ds160 form",
    "how long does visa processing take",
]

for q in test_queries:
    results = collection.query(
        query_texts=[q],
        n_results=3,
        include=["metadatas", "distances"]
    )
    print(f"\n🔍 '{q}'")
    for i in range(len(results["ids"][0])):
        dist = results["distances"][0][i]
        meta = results["metadatas"][0][i]
        relevance = "✅" if dist < 0.4 else "⚠️"
        print(f"  {relevance} [{dist:.3f}] {meta['category']:<30} {meta['heading'][:50]}")


🔍 'what documents do i need for b2 tourist visa'
  ⚠️ [0.492] ds160_application              What documents do I need to have with me while I c
  ⚠️ [0.529] visa_types                     Nonimmigrant Visa Categories
  ⚠️ [0.533] photo_requirements             Photo Requirements

🔍 'how much is the visa application fee'
  ✅ [0.222] visa_fees                      Description of Service and Fee Amount (All fees = 
  ✅ [0.286] visa_fees                      Description of Service and Fee Amount (All fees = 
  ✅ [0.315] visa_fees                      Coming to the United States Permanently - Immigran

🔍 'what happens if my visa is denied'
  ✅ [0.300] visa_denials                   Visa Denials
  ✅ [0.399] visa_denials                   Visa Denials
  ⚠️ [0.430] visa_denials                   INA Section 212(a)(9)(B)(i) - Unlawful Presence in

🔍 'how to fill ds160 form'
  ✅ [0.326] visa_process_ustraveldocs      DS-160 information
  ✅ [0.327] visa_process_ustraveldocs      Guidelines for c

In [25]:
# Improvement 1: check what the b2 query is actually returning (is it useful?)
results = collection.query(
    query_texts=["what documents do i need for b2 tourist visa"],
    n_results=2,
    include=["documents", "metadatas", "distances"]
)
for i in range(2):
    print(f"\n[{i+1}] {results['metadatas'][0][i]['heading']}")
    print(results['documents'][0][i][:400])
    print("─"*60)


[1] What documents do I need to have with me while I complete the DS-160?
What documents do I need to have with me while I complete the DS-160?
You should have the following documents available while you complete your DS-160:
Travel itinerary, if you have already made travel arrangements. Dates of your last five visits or trips to the United States, if you have previously travelled to the United States. You may also be asked for your international travel history for the
────────────────────────────────────────────────────────────

[2] Nonimmigrant Visa Categories
Nonimmigrant Visa Categories
Purpose of Travel: Tourism, vacation, pleasure visitor | Visa Category: B-2 | Required: Before applying for visa*: (NA)
────────────────────────────────────────────────────────────


In [26]:
from dotenv import load_dotenv
import os
load_dotenv()


True

In [27]:
groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])

SYSTEM_PROMPT = """You are a helpful US visa assistant specialising in non-immigrant visas 
for applicants based in India. Answer the user's question using ONLY the context passages 
provided below.
Rules:
- Be concise and factual. Use bullet points where helpful.
- If the context does not contain enough information, say so honestly — do not guess.
- Always mention the relevant visa category when applicable (e.g. B-1/B-2, F-1, H-1B).
- End your answer with a Sources section listing the unique source URLs used."""

def ask(question: str, top_k: int = 6) -> str:
    # 1. Retrieve
    results = collection.query(
        query_texts=[question],
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )

    # 2. Filter by relevance and build context
    context_blocks = []
    sources = []
    for i in range(len(results["ids"][0])):
        dist = results["distances"][0][i]
        if dist > 0.55:   # skip anything too distant
            continue
        doc  = results["documents"][0][i]
        meta = results["metadatas"][0][i]
        context_blocks.append(
            f"[{i+1}] Category: {meta['category']} | Heading: {meta['heading']}\n{doc}"
        )
        if meta["source_url"] not in sources:
            sources.append(meta["source_url"])

    if not context_blocks:
        return "I couldn't find relevant information for that question."

    context = "\n\n".join(context_blocks)
    user_prompt = f"Context:\n{context}\n\nQuestion: {question}"

    # 3. Generate
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=0.2,
        max_tokens=1024,
    )
    return response.choices[0].message.content.strip()


# ── Test it ────────────────────────────────────────────────────────────────────
test_questions = [
    "what documents do i need for a b2 tourist visa?",
    "how much is the MRV fee?",
    "what happens if my visa is denied under 214b?",
]

for q in test_questions:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    print(f"{'─'*60}")
    print(ask(q))


Q: what documents do i need for a b2 tourist visa?
────────────────────────────────────────────────────────────
For a B-2 tourist visa, you will need to have the following documents available while completing the DS-160:
* Travel itinerary, if you have already made travel arrangements
* Dates of your last five visits or trips to the United States, if you have previously traveled to the United States
* Résumé or Curriculum Vitae - You may be required to provide information about your current and previous education and work history
* Other Information - Some applicants, depending on the intended purpose of travel, will be asked to provide additional information when completing the DS-160

You will also need to provide a digital image or photo as part of your visa application. The acceptance of your digital image or photo is at the discretion of the U.S. embassy or consulate where you apply.

Sources:
[1] https://www.ustraveldocs.com/in/in-niv-ds160application.asp 
(Note: The actual URL 

In [28]:
def ask(question: str, top_k: int = 8) -> str:
    # 1. Retrieve broadly
    results = collection.query(
        query_texts=[question],
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )

    print(f"\n===== NOTEBOOK RETRIEVAL: '{q}' =====")

    for i in range(len(results["ids"][0])):
        dist = results["distances"][0][i]
        meta = results["metadatas"][0][i]
        doc  = results["documents"][0][i]

        print(f"{round(dist,3)} | {meta['category']} | {meta['heading']}")
        print(doc[:200])
        print("-" * 50)
    # 2. Also do a targeted fee lookup if question is about fees/cost/amount
    fee_keywords = ["fee", "cost", "how much", "price", "mrv", "payment", "amount"]
    is_fee_query = any(kw in question.lower() for kw in fee_keywords)

    bonus_chunks = []
    if is_fee_query:
        fee_results = collection.query(
            query_texts=["visa application fee amount USD dollars"],
            n_results=4,
            include=["documents", "metadatas", "distances"],
            where={"category": "visa_fees"}   # only look in fee chunks
        )
        for doc, meta, dist in zip(
            fee_results["documents"][0],
            fee_results["metadatas"][0],
            fee_results["distances"][0],
        ):
            bonus_chunks.append((doc, meta, dist))

    # 3. Merge main + bonus results
    context_blocks = []
    sources = []
    seen_docs = set()

    all_results = list(zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    )) + bonus_chunks

    for doc, meta, dist in all_results:
        if dist > 0.65:
            continue
        fp = doc[:100]
        if fp in seen_docs:   # deduplicate
            continue
        seen_docs.add(fp)

        context_blocks.append(
            f"Category: {meta['category']} | Heading: {meta['heading']}\n{doc}"
        )
        url = meta["source_url"]
        if url and url not in sources:
            sources.append(url)

    if not context_blocks:
        return "I couldn't find relevant information for that question."

    sources_text = "\n".join(f"- {url}" for url in sources)
    context      = "\n\n".join(context_blocks)
    user_prompt  = (
        f"Context:\n{context}\n\n"
        f"Real source URLs (use ONLY these, do not invent any):\n{sources_text}\n\n"
        f"Question: {question}"
    )

    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=0.2,
        max_tokens=1024,
    )
    return response.choices[0].message.content.strip()


# Test just the fee question
print(ask("how much is the MRV fee?"))
print("\n---\n")
print(ask("what is the visa application fee for b1 b2?"))


===== NOTEBOOK RETRIEVAL: 'what happens if my visa is denied under 214b?' =====
0.39 | visa_process_ustraveldocs | Pay your visa fees
Pay your visa fees
All nonimmigrant visa application fee (also known as the MRV fee) payments made on or after October 1, 2022, are valid for 365 days from the date a receipt is issued for payment of 
--------------------------------------------------
0.435 | visa_process_ustraveldocs | Bank and payment options
Bank and payment options

In most cases, each visa applicant, including children, is required to pay a non-refundable, non-transferable Machine Readable Visa (MRV) application fee, whether a visa is i
--------------------------------------------------
0.47 | visa_fees | Coming to the United States Permanently - Immigrant Services
Coming to the United States Permanently - Immigrant Services
Diversity Visa Registration Fee (paid at time of registration only by the principal applicant): Diversity Visa Application Fee (per person 
-------------------

In [29]:
# Inspect what the fee chunks actually contain
results = collection.query(
    query_texts=["MRV fee nonimmigrant visa application fee amount"],
    n_results=5,
    include=["documents", "metadatas", "distances"]
)

for i in range(len(results["ids"][0])):
    meta = results["metadatas"][0][i]
    doc  = results["documents"][0][i]
    print(f"\n[{i+1}] dist={results['distances'][0][i]:.3f} | {meta['category']} | {meta['content_type']}")
    print(f"Heading: {meta['heading']}")
    print(f"Text:\n{doc}")
    print("─"*60)


[1] dist=0.205 | visa_process_ustraveldocs | 
Heading: Pay your visa fees
Text:
Pay your visa fees
All nonimmigrant visa application fee (also known as the MRV fee) payments made on or after October 1, 2022, are valid for 365 days from the date a receipt is issued for payment of the MRV fee. Applicants must schedule an in-person or interview waiver appointment during this 365-day period. Please note applicants must only schedule their in-person or interview waiver appointment within the 365-day period. There is no requirement the interview or VAC appointment must occur during the 365-day period. All receipts for payment of MRV fees issued before October 1, 2022, were extended until September 30, 2023, and remain valid until this date.
────────────────────────────────────────────────────────────

[2] dist=0.220 | visa_process_ustraveldocs | 
Heading: Pay your visa fees
Text:
Pay your visa fees
Visa applicants, including children, are required to pay a non-refundable, non-transferable v

In [30]:
print(ask("what should be the minimum bank balance for 18+ students applying for US visaa"))


===== NOTEBOOK RETRIEVAL: 'what happens if my visa is denied under 214b?' =====
0.51 | visa_process_ustraveldocs | Fee: $185 USD
Fee: $185 USD
Visitor visas are nonimmigrant visas for persons who want to enter the United States temporarily for business (visa category B-1), for tourism (visa category B-2), or for a combination o
--------------------------------------------------
0.527 | visa_process_ustraveldocs | Bank and payment options
Bank and payment options

In most cases, each visa applicant, including children, is required to pay a non-refundable, non-transferable Machine Readable Visa (MRV) application fee, whether a visa is i
--------------------------------------------------
0.538 | visa_process_ustraveldocs | SEVIS Fee
SEVIS Fee
F, M and J visa principal applicants: Check with your U.S. school to make sure your information has been entered into SEVIS. You will need to pay a separate SEVIS fee in addition to the visa
--------------------------------------------------
0.546 |

In [31]:
print(ask("what is the fees for B1/B2 visa?"))


===== NOTEBOOK RETRIEVAL: 'what happens if my visa is denied under 214b?' =====
0.267 | visa_process_ustraveldocs | Fee: $185 USD
Fee: $185 USD
Visitor visas are nonimmigrant visas for persons who want to enter the United States temporarily for business (visa category B-1), for tourism (visa category B-2), or for a combination o
--------------------------------------------------
0.318 | visa_fees | Description of Service and Fee Amount (All fees = $ in US currency)
Description of Service and Fee Amount (All fees = $ in US currency)
*Though petition-based nonimmigrant visas, the processing fee for these visas is $185.00
Petition based visa categories: $205.00
Inc
--------------------------------------------------
0.32 | visa_fees | Description of Service and Fee Amount (All fees = $ in US currency)
Description of Service and Fee Amount (All fees = $ in US currency)
B: M | Visitor Visa: Business, Tourism, Medical treatment: Students, Vocational
------------------------------------------

In [32]:
print(ask("what are the required documents i need to take for my VAC"))


===== NOTEBOOK RETRIEVAL: 'what happens if my visa is denied under 214b?' =====
0.496 | ds160_application | What documents do I need to have with me while I complete the DS-160?
What documents do I need to have with me while I complete the DS-160?
You should have the following documents available while you complete your DS-160:
Travel itinerary, if you have already made trave
--------------------------------------------------
0.51 | visa_process_ustraveldocs | Other Information
Other Information
All persons that enter the VAC site are subject to a security inspection.
The use of cell phones and cameras are prohibited.
All bags such as travel bags, smart watches, flash drives
--------------------------------------------------
0.541 | visa_process_ustraveldocs | Visa Application Centers (VAC)
Visa Application Centers (VAC)
Most applicants will be required to have an appointment at a Visa Application Center (VAC) prior to their appointment at the Consular Sections. Biometric information 

reranker, keyword matching bm25

cicd pipeline

In [33]:
print(ask("what is the fees for B1/B2 visa?"))


===== NOTEBOOK RETRIEVAL: 'what happens if my visa is denied under 214b?' =====
0.267 | visa_process_ustraveldocs | Fee: $185 USD
Fee: $185 USD
Visitor visas are nonimmigrant visas for persons who want to enter the United States temporarily for business (visa category B-1), for tourism (visa category B-2), or for a combination o
--------------------------------------------------
0.318 | visa_fees | Description of Service and Fee Amount (All fees = $ in US currency)
Description of Service and Fee Amount (All fees = $ in US currency)
*Though petition-based nonimmigrant visas, the processing fee for these visas is $185.00
Petition based visa categories: $205.00
Inc
--------------------------------------------------
0.32 | visa_fees | Description of Service and Fee Amount (All fees = $ in US currency)
Description of Service and Fee Amount (All fees = $ in US currency)
B: M | Visitor Visa: Business, Tourism, Medical treatment: Students, Vocational
------------------------------------------